In [0]:
data = spark.read.table("`01_bronze`.raw.customers")
display(data)

In [0]:
import pyspark.sql.functions as F

def standardize_date(columnName, df):
    return df.withColumn(
        "date_parsed",
        F.coalesce(
            F.try_to_date(F.col(columnName), "M/d/yyyy"),
            F.try_to_date(F.col(columnName), "M-d-yyyy"),
            F.try_to_date(F.col(columnName), "yyyy-M-d"),
            F.try_to_date(F.col(columnName), "yyyy/M/d")
        )
    ).withColumn(
        columnName,
        F.trim(F.col("date_parsed")).cast('date')
    ).drop("date_parsed")

data = standardize_date("birthday", data)
display(data)



In [0]:
def state_null_handle(df):

    mapping_df = df.filter(F.col("state").isNotNull())\
                   .select("state_code", "state")\
                   .distinct()
    return df.alias("a").join(
        mapping_df.alias("b"), 
        on="state_code", 
        how="left"
    ).select(
        F.col("a.customerkey"),
        F.col("a.gender"),
        F.col("a.name"),
        F.col("a.city"),
        F.col("a.state_code"),
        F.coalesce(F.col("a.state"), F.col("b.state")).alias("state"),
        F.col("a.zip_code"),
        F.col("a.country"),
        F.col("a.continent"),
        F.col("a.birthday")
    )

data = state_null_handle(data)
display(data)


In [0]:
def state_null_handle(df):
    mapping_df = df.filter(F.col("zip_code").isNotNull())\
                   .select("state_code", "zip_code")\
                   .distinct()
    return df.alias("a").join(
        mapping_df.alias("b"), 
        on="zip_code", 
        how="left"
    ).select(
        F.col("a.customerkey"),
        F.col("a.gender"),
        F.col("a.name"),
        F.col("a.city"),
        F.col("a.state_code"),
        F.col("a.state"),
        F.coalesce(F.col("a.zip_code"), F.col("b.zip_code")).alias("zip_code"),
        F.col("a.country"),
        F.col("a.continent"),
        F.col("a.birthday")
    )

data = state_null_handle(data)
display(data)

In [0]:
data = data.filter(F.col('customerkey').isNotNull())
display(data)

In [0]:
def dataTypeHandle(df,colname,dataType):
    if dict(df.dtypes)[colname] == 'string':
        return df.withColumn(colname,F.trim(F.col(colname)).try_cast(dataType))
    if dict(df.dtypes)[colname] == dataType:
        return df.withColumn(colname,F.col(colname))

temp = data
temp = dataTypeHandle(temp,'customerkey','int')
temp = dataTypeHandle(temp,'gender','string')
temp = dataTypeHandle(temp,'name','string')
temp = dataTypeHandle(temp,'city','string')
temp = dataTypeHandle(temp,'state_code','string')
temp = dataTypeHandle(temp,'state','string')
temp = dataTypeHandle(temp,'zip_code','int')
temp = dataTypeHandle(temp,'country','string')
temp = dataTypeHandle(temp,'continent','string')
temp = dataTypeHandle(temp,'birthday','date')

temp.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in temp.columns]).show()